In [1]:
# Install packages
!pip3 install transformers==4.45.2 sentencepiece==0.2.2 torch==2.5.1
!pip3 install datasets pandas
SEED = 7

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [2]:
# import opus translation model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

name = "Helsinki-NLP/opus-mt-nl-en"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSeq2SeqLM.from_pretrained(name)

/Users/jaridzatromp/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/jaridzatromp/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/jaridzatromp/Library/Python/3.9/lib/python/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [4]:
from huggingface_hub import HfApi

api = HfApi()
for repo in ["Helsinki-NLP/opus-mt-nl-en",
             "yunvs/bart-base-wikilarge-simplification",
             "BramVanroy/ul2-base-dutch-simplification-mai-2023"]:
    print(repo, api.model_info(repo).sha)

from huggingface_hub import HfApi
print(HfApi().dataset_info("igornishka/dutch-municipal-sentence-simplification").sha)

Helsinki-NLP/opus-mt-nl-en 48af999f2c59b10c05ca6e008dcedc07677a9b15
yunvs/bart-base-wikilarge-simplification 050112d6e53fbde261fdc120ce19a687a9e13086
BramVanroy/ul2-base-dutch-simplification-mai-2023 4ed2e70657f452c3bed13f7c64f126220c3b6450
5d1fd646a0aeda9dba789d3b7f4284a5b1e70266


In [5]:
MODELS = {
    "mt":    ("Helsinki-NLP/opus-mt-nl-en", "48af999f2c59b10c05ca6e008dcedc07677a9b15"),
    "en_ts": ("yunvs/bart-base-wikilarge-simplification", "050112d6e53fbde261fdc120ce19a687a9e13086"),
    "nl_ts": ("BramVanroy/ul2-base-dutch-simplification-mai-2023", "050112d6e53fbde261fdc120ce19a687a9e13086"),
}

name, rev = MODELS["mt"]
tokenizer = AutoTokenizer.from_pretrained(name, revision=rev)
model = AutoModelForSeq2SeqLM.from_pretrained(name, revision=rev)

/Users/jaridzatromp/Library/Python/3.9/lib/python/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [10]:
!pip3 list | grep torch

torch                         2.5.1
torchvision                   0.19.1
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [6]:
from datasets import load_dataset

ds = load_dataset(
    "igornishka/dutch-municipal-sentence-simplification",
    revision="5d1fd646a0aeda9dba789d3b7f4284a5b1e70266", 
)
df = ds["train"].to_pandas()
print(df.columns)   # ['Document ID', 'Complex', 'Simple']
print(len(df))

Generating train split: 1310 examples [00:00, 75907.14 examples/s]

Index(['Document ID', 'Complex', 'Simple'], dtype='object')
1310


In [8]:
# Load data from SimPA
import pandas as pd
from pathlib import Path

def read_lines(p):
    return Path(p).read_text(encoding="utf-8").splitlines()

d = "data/simpa/"

# Syntactic: 1 reference per sentence
ss = pd.DataFrame({"src": read_lines(d + "ss.original"),
                   "ss_ref": read_lines(d + "ss.simplified")})

# Lexical: 3 references per sentence, grouped by original
ls = pd.DataFrame({"src": read_lines(d + "ls.original"),
                   "ls_ref": read_lines(d + "ls.simplified")})
ls_refs = ls.groupby("src", sort=False)["ls_ref"].apply(list)

simpa = ss.join(ls_refs, on="src")
print(len(simpa), simpa["ls_ref"].str.len().value_counts())   # expect 1100, all 3

1100 ls_ref
3.0    876
Name: count, dtype: int64


In [9]:
# Dataset statistics
def stats(src, ref, name):
    s_len = src.str.split().str.len()
    r_len = ref.str.split().str.len()
    return {
        "dataset": name,
        "pairs": len(src),
        "src tokens/sent": round(s_len.mean(), 1),
        "ref tokens/sent": round(r_len.mean(), 1),
        "compression": round((r_len / s_len).mean(), 2),
        "% unchanged": round((src.str.strip() == ref.str.strip()).mean() * 100, 1),
    }

table = pd.DataFrame([
    stats(df["Complex"], df["Simple"], "Amsterdam (nl)"),
    stats(simpa["src"], simpa["ss_ref"], "SimPA (en, ss)"),
])
print(table.to_latex(index=False))   # paste straight into Overleaf

\begin{tabular}{lrrrrr}
\toprule
dataset & pairs & src tokens/sent & ref tokens/sent & compression & % unchanged \\
\midrule
Amsterdam (nl) & 1310 & 16.300000 & 15.100000 & 0.970000 & 0.800000 \\
SimPA (en, ss) & 1100 & 27.800000 & 26.200000 & 0.950000 & 1.100000 \\
\bottomrule
\end{tabular}



## Human evaluation sample

In [1]:
import importlib

missing = [pkg for pkg in ("datasets", "pandas") if importlib.util.find_spec(pkg) is None]
if missing:
    !pip3 install -q {' '.join(missing)}

SEED = 42
N = 60
DATASET = "igornishka/dutch-municipal-sentence-simplification"
REVISION = "5d1fd646a0aeda9dba789d3b7f4284a5b1e70266"  # pinned commit, do not change

In [2]:
from datasets import load_dataset
import pandas as pd

raw = load_dataset(DATASET, revision=REVISION)["train"].to_pandas()
raw["row_id"] = raw.index  # original row index, so every sentence stays traceable

print(f"total pairs: {len(raw)}")
print(f"unique documents: {raw['Document ID'].nunique()}")

total pairs: 1310
unique documents: 48


In [3]:
clean = raw.copy()
clean["Complex"] = clean["Complex"].str.strip()
clean["Simple"] = clean["Simple"].str.strip()
print(f"start: {len(clean)}")

clean = clean[clean["Complex"].str.split().str.len() >= 12]
print(f"after min 12 whitespace tokens: {len(clean)}")

clean = clean[clean["Complex"].str.match(r".*[.?!]$", na=False)]
print(f"after ends in . ? !: {len(clean)}")

GAP_PATTERN = r"\s,|naar \.|\s\.$|via\.$"
clean = clean[~clean["Complex"].str.contains(GAP_PATTERN, regex=True, na=False)]
print(f"after removing gaps from stripped emails/URLs: {len(clean)}")

clean = clean[~clean["Complex"].str.contains(r"\[", regex=True, na=False)]
print(f"after removing anonymisation placeholders: {len(clean)}")

clean = clean[clean["Complex"] != clean["Simple"]]
print(f"after Complex != Simple: {len(clean)}")

clean = clean.drop_duplicates(subset="Complex")
print(f"after dropping duplicate Complex sentences: {len(clean)}")

start: 1310
after min 12 whitespace tokens: 930
after ends in . ? !: 858
after removing gaps from stripped emails/URLs: 848
after removing anonymisation placeholders: 823
after Complex != Simple: 823
after dropping duplicate Complex sentences: 814


In [4]:
CAP = 3
capped = clean.sample(frac=1, random_state=SEED).groupby("Document ID").head(CAP)

if len(capped) < N:
    CAP = 4
    print(f"fewer than {N} rows after capping at 3 per document; raising cap to {CAP}")
    capped = clean.sample(frac=1, random_state=SEED).groupby("Document ID").head(CAP)

sample = capped.sample(n=N, random_state=SEED).sort_values("row_id").reset_index(drop=True)

sample = sample.rename(columns={
    "Document ID": "doc_id",
    "Complex": "complex_nl",
    "Simple": "simple_nl",
})
sample["item_id"] = [f"s{i:02d}" for i in range(1, N + 1)]
sample = sample[["item_id", "row_id", "doc_id", "complex_nl", "simple_nl"]]

sample.to_csv("human_eval_sample.csv", index=False)
print(f"saved {len(sample)} rows to human_eval_sample.csv")

saved 60 rows to human_eval_sample.csv


In [5]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.max_rows", None)

tokens_per_sentence = sample["complex_nl"].str.split().str.len()

print(f"documents in sample: {sample['doc_id'].nunique()}")
print("sentences per document:")
print(sample.groupby("doc_id").size())
print(f"tokens per sentence - mean: {tokens_per_sentence.mean():.1f}, "
      f"min: {tokens_per_sentence.min()}, max: {tokens_per_sentence.max()}")

sample

documents in sample: 36
sentences per document:
doc_id
1     2
4     3
5     1
6     1
7     1
9     1
11    1
13    3
14    1
16    1
17    1
18    2
20    2
21    3
22    1
24    2
25    1
26    2
28    2
29    1
30    2
31    1
32    2
33    2
34    1
35    3
36    1
38    2
39    3
41    1
42    1
43    2
44    1
46    2
47    1
48    3
dtype: int64
tokens per sentence - mean: 18.7, min: 12, max: 38


,item_id,row_id,doc_id,complex_nl,simple_nl
0,s01,3,1,Ook komt naar voren dat verschillende bewoners niet bekend zijn met huidige vervoersalternatieven zoals de Buurtbus of niet weten hoe ze hiervan gebruik kunnen maken.,Of niet weten hoe ze dat kunnen gebruiken.
1,s02,4,1,"Daarnaast geven de meeste bewoners aan buslijn 21 terug te willen in de buurt, of een betere verbinding met de dichtstbijzijnde bushalte.",Daarnaast willen de meeste bewoners buslijn 21 terug in de buurt. Of een betere verbinding met de dichtstbijzijnde bushalte.
2,s03,11,4,"ma t/m vrij: 17:00 tot 08:00 uur, za en zo en nat.",maandag tot en met vrijdag: 17:00 tot 08:00 uur
3,s04,12,4,"GGD Amsterdam: 020-55 55 202 ma t/m vrij: 09.00 - 18.00 uur, za en zo 10.00 -17.00 uur.",GGD Amsterdam: 0205 5 55 202 ma t/m vrij: 09.00 - 18.00 uur
4,s05,16,4,"Pak het vuile wasgoed (alleen in gesloten vuilniszak), de vuile vaat, afval etc.","Pak het vuile wasgoed (alleen in gesloten vuilniszak), de vuile vaat, afval enzovoort van de kamer dat bij de deur staat."
5,s06,21,5,"Door het coronavirus worden er extra maatregelen genomen, zodat u coronaproof/veilig kunt stemmen.",Door het coronavirus hebben we extra maatregelen zodat u veilig kunt stemmen.
6,s07,35,6,Dat betekent dat werkzaamheden voor Weesp zoveel mogelijk in Weesp moeten worden uitgevoerd.,Dat betekent dat werkzaamheden voor Weesp zoveel mogelijk in Weesp moeten gebeuren.
7,s08,75,7,Dan kan het zijn dat u tijdens uw arbeidsongeschiktheid reiskosten maakt in verband met uw re-integratie.,"Als u langer dan 6 weken volledig arbeidsongeschikt bent, maakt u misschien reiskosten tijdens uw arbeidsongeschiktheid vanwege uw re-integratie."
8,s09,85,9,"Uw leidinggevende heeft aangegeven dat u op grond van artikel 7.1, lid 7 PGA bij uw langdurige verzuim uw volledige salaris en salaristoelagen blijft houden.",Uw leidinggevende gaf aan dat u bij uw langdurige verzuim uw volledige salaris en salaristoelagen blijft houden.
9,s10,91,11,Met zonnepanelen op uw dak wekt u uw eigen duurzame energie op en bespaart u gemiddeld tussen de 300 en 600 euro per jaar op uw energierekening.,"Met zonnepanelen op uw dak wekt u uw eigen duurzame energie op. U bespaart daarmee gemiddeld tussen de € 300,- en € 600,- per jaar op uw energierekening."
